# Create flag parameter

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
# creating utilities

dbutils.widgets.text('incremental_flag', '0')

In [0]:
# check if the table exists with any data then 
if spark.catalog.tableExists('cars_catalog.gold.dim_date'):
    if (spark.sql("SELECT MAX(dim_date_key) FROM cars_catalog.gold.dim_date").collect()[0][0] > 0):
        incremental_flag = '1'
else:
    incremental_flag = dbutils.widgets.get('incremental_flag')

print(incremental_flag)

1


# creating dimension Model


### Fetch relative columns

In [0]:
# spark.sql('''
#                       SELECT *
#                       FROM parquet.`abfss://silver@adlsforde.dfs.core.windows.net/carsales`
#                       ''').display()

In [0]:
source_df = spark.sql('''
                      SELECT DISTINCT(DATE_ID) as date_id
                      FROM parquet.`abfss://silver@adlsforde.dfs.core.windows.net/carsales`
                      ''')



In [0]:
source_df.display()

date_id
DT00029
DT00030
DT00039
DT00078
DT00116
DT00120
DT00124
DT00136
DT00140
DT00164



### Dim Date Sink initial and incremental (Just Bring the schema if table not exitsis)

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_date'):
    sink_df = spark.sql('''
                        SELECT dim_date_key, date_id
                        FROM cars_catalog.gold.dim_date
                        ''')
# Initial dimention table creation
else:
    sink_df = spark.sql('''
                        SELECT 1 as dim_date_key,  DATE_ID as date_id
                    FROM parquet.`abfss://silver@adlsforde.dfs.core.windows.net/carsales`
                    WHERE 1 = 0
                    ''')


### Fintering new records and old records

In [0]:
filter_df = source_df.join(sink_df, source_df.date_id == sink_df.date_id, 'left')\
    .select(source_df.date_id, sink_df.dim_date_key)

**df_filter_old**

In [0]:
df_filter_old = filter_df.filter(col('dim_date_key').isNotNull())


**df_filter_new**

In [0]:
df_filter_new = filter_df.filter(col('dim_date_key').isNull()).select('date_id')


### Create Surrogate Key
**Fetch the max surrogate key from existing dim table**

In [0]:
if (incremental_flag == '0'):
    max_value = 1
else:
    max_value_df = spark.sql("SELECT MAX(dim_date_key) FROM cars_catalog.gold.dim_date")
    max_value = max_value_df.collect()[0][0]




**Create Surrogate key column and add the max surrogate key**

In [0]:
df_filter_new = df_filter_new.withColumn('dim_date_key', max_value + monotonically_increasing_id())
df_filter_new.display()

date_id,dim_date_key


### Create Final data frame - df_filter_old + df_filter_new


In [0]:
final_df =  df_filter_new.union(df_filter_old)

In [0]:
final_df.display()

date_id,dim_date_key
DT00029,1156
DT00030,1157
DT00039,1158
DT00078,1159
DT00116,1160
DT00120,1161
DT00124,1162
DT00136,1163
DT00140,1164
DT00164,1165


### SCD Type - 1 (UPSERT)
**Update(existing change data) + Insert(new data)**

In [0]:
# Incremental Run
if spark.catalog.tableExists("cars_catalog.gold.dim_date"):
    delta_table = DeltaTable.forPath(spark, "abfss://gold@adlsforde.dfs.core.windows.net/dim_date")
    
    delta_table.alias('trg').merge(final_df.alias('src'), "trg.dim_date_key = src.dim_date_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()
# initial run
else:
    final_df.write.format('delta')\
        .mode("overwrite")\
        .option("path", "abfss://gold@adlsforde.dfs.core.windows.net/dim_date")\
        .saveAsTable("cars_catalog.gold.dim_date")

In [0]:
%sql
SELECT * FROM cars_catalog.gold.dim_date

date_id,dim_date_key
DT00029,1
DT00030,2
DT00039,3
DT00078,4
DT00116,5
DT00120,6
DT00124,7
DT00136,8
DT00140,9
DT00164,10
